# 04 - Five-split faithfulness and stability of explanations


**What this notebook does -**

(1) Repeats the Integrated Gradients faithfulness test over five independent training runs (seeds 1-5, each with its own stratified 80/20 split) and compares with a highest-pressure baseline using deletion and insertion AUC.

(2) Trains five CNNs that differ only in their random seed and measures how much their explanations agree (sensor rankings, top-10% overlap, CAM curves), with a control.


## Setup

In [ ]:
import os
if not os.path.exists("/content/project"):
    !git clone https://github.com/BHAGI-99/CS5998-Capstone-Project-P.B.W.Nanayakkara-268472T.git /content/project
%cd /content/project
!git pull
!python src/download_data.py
!python src/segment.py

/content/project
Already up to date.
Already downloaded: data/raw/Smart-Sleeve-Dataset
X (2520, 208, 200) uint16 | classes: 18 | subjects: 14
truncated segments: 123


## 1. Five-split faithfulness experiment
Lower AUC is better for deletion and higher is better for insertion.

In [ ]:
import os; os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd, tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers

P = Path("data/processed"); R = Path("results"); R.mkdir(exist_ok=True)
X = np.load(P/"X.npy").astype(np.float32) / 4095.0
y = np.load(P/"y.npy"); L = np.minimum(np.load(P/"lengths.npy"), 208)
SEEDS, EPOCHS = [1, 2, 3, 4, 5], 60
K2 = [0, 1, 2, 3, 5, 7, 10, 15, 20]          # % of the 200 sensors removed / inserted

def set_seed(s):
    np.random.seed(s); tf.random.set_seed(s); tf.keras.utils.set_random_seed(s)

def build_cnn(seq_len=208, n_feat=200, n_classes=18):
    inp = tf.keras.Input(shape=(seq_len, n_feat))
    x = layers.Conv1D(64, 5, padding="same")(inp);  x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv1D(128, 5, padding="same")(x);   x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv1D(128, 3, padding="same", name="last_conv")(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling1D()(x); x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def auc2(c):
    k = np.array(K2, float); return float(((c[:-1] + c[1:]) / 2 * np.diff(k)).sum() / (k[-1] - k[0]))

def run_seed(seed):
    set_seed(seed)
    tr, te = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=seed)
    tr_in, va = train_test_split(tr, test_size=0.1, stratify=y[tr], random_state=seed)
    model = build_cnn()
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    model.fit(X[tr_in], y[tr_in], validation_data=(X[va], y[va]), epochs=EPOCHS, batch_size=32, callbacks=[es], verbose=0)
    pred = model.predict(X[te], verbose=0).argmax(1); acc = accuracy_score(y[te], pred)
    base = np.mean([X[i, :L[i]].mean(0) for i in tr_in], axis=0)   # typical pressure of each sensor

    def ig(x, cls, steps=50):                                  # Integrated Gradients, zero baseline
        x = tf.constant(x[None]); b = tf.zeros_like(x)
        a = tf.reshape(tf.linspace(0.0, 1.0, steps + 1), (-1, 1, 1)); inter = b + a * (x - b)
        with tf.GradientTape() as tape:
            tape.watch(inter); p = model(inter, training=False)[:, cls]
        g = tape.gradient(p, inter); g = (g[:-1] + g[1:]) / 2.0
        return ((x - b)[0] * tf.reduce_mean(g, axis=0)).numpy()

    sens_ig = np.zeros((len(te), 200)); sens_pr = np.zeros((len(te), 200))
    for j, i in enumerate(te):
        sens_ig[j] = np.abs(ig(X[i], int(pred[j]))[:L[i]]).mean(0); sens_pr[j] = X[i, :L[i]].mean(0)

    def make(sens, k, mode):
        n = int(round(200 * k / 100)); Xm = X[te].copy(); top = np.argsort(-sens, axis=1)[:, :n]
        for j in range(len(te)):
            Lj = L[te][j]
            if mode == "delete":
                if n > 0: Xm[j, :Lj][:, top[j]] = base[top[j]]
            else:
                Xm[j, :Lj] = base
                if n > 0: Xm[j, :Lj][:, top[j]] = X[te][j, :Lj][:, top[j]]
        return Xm
    def curve(sens, mode):
        return np.array([accuracy_score(y[te], model.predict(make(sens, k, mode), verbose=0).argmax(1)) for k in K2])

    row = {"seed": seed, "test_acc": round(acc, 4)}
    for mode in ["delete", "insert"]:
        row[f"{mode}_IG"] = auc2(curve(sens_ig, mode)); row[f"{mode}_pressure"] = auc2(curve(sens_pr, mode))
        row[f"{mode}_random"] = np.mean([auc2(curve(np.random.default_rng(s).random((len(te), 200)), mode)) for s in range(2)])
    return row

rows = []
for s in SEEDS:
    r = run_seed(s); rows.append(r); print({k: (round(float(v), 3)) for k, v in r.items()})
df = pd.DataFrame(rows).round(4); df.to_csv(R/"xai_faithfulness_multiseed.csv", index=False)
d_del = df.delete_pressure - df.delete_IG          # positive = IG deletes faster (better)
d_ins = df.insert_IG - df.insert_pressure          # positive = IG inserts faster (better)
print("\nMEAN +/- SD over seeds")
print(df[["test_acc","delete_IG","delete_pressure","delete_random","insert_IG","insert_pressure","insert_random"]].agg(["mean","std"]).round(3).to_string())
print(f"\nDeletion: pressure AUC minus IG AUC = {d_del.mean():+.3f} (SD {d_del.std():.3f}); IG better in {(d_del>0).sum()} of {len(df)} seeds")
print(f"Insertion: IG AUC minus pressure AUC = {d_ins.mean():+.3f} (SD {d_ins.std():.3f}); IG better in {(d_ins>0).sum()} of {len(df)} seeds")

{'seed': 1.0, 'test_acc': 0.696, 'delete_IG': 0.241, 'delete_pressure': 0.254, 'delete_random': 0.645, 'insert_IG': 0.302, 'insert_pressure': 0.295, 'insert_random': 0.067}
{'seed': 2.0, 'test_acc': 0.693, 'delete_IG': 0.264, 'delete_pressure': 0.275, 'delete_random': 0.643, 'insert_IG': 0.314, 'insert_pressure': 0.325, 'insert_random': 0.071}
{'seed': 3.0, 'test_acc': 0.732, 'delete_IG': 0.266, 'delete_pressure': 0.277, 'delete_random': 0.681, 'insert_IG': 0.366, 'insert_pressure': 0.348, 'insert_random': 0.078}
{'seed': 4.0, 'test_acc': 0.734, 'delete_IG': 0.246, 'delete_pressure': 0.249, 'delete_random': 0.678, 'insert_IG': 0.322, 'insert_pressure': 0.331, 'insert_random': 0.077}
{'seed': 5.0, 'test_acc': 0.714, 'delete_IG': 0.249, 'delete_pressure': 0.267, 'delete_random': 0.672, 'insert_IG': 0.364, 'insert_pressure': 0.354, 'insert_random': 0.093}

MEAN +/- SD over seeds
      test_acc  delete_IG  delete_pressure  delete_random  insert_IG  insert_pressure  insert_random
mean     0

## Summary of the five-split result

In [ ]:
import pandas as pd
df = pd.read_csv("results/xai_faithfulness_multiseed.csv")
print(df.to_string(index=False))
d_del = df.delete_pressure - df.delete_IG                   # positive = IG deletes faster (better)
d_ins = df.insert_IG - df.insert_pressure                   # positive = IG inserts faster (better)
print("\nMEAN +/- SD over seeds")
print(df[["test_acc","delete_IG","delete_pressure","delete_random","insert_IG","insert_pressure","insert_random"]].agg(["mean","std"]).round(3).to_string())
print(f"\nDeletion: pressure AUC minus IG AUC = {d_del.mean():+.3f} (SD {d_del.std():.3f}); IG better in {(d_del>0).sum()} of {len(df)} seeds")
print(f"Insertion: IG AUC minus pressure AUC = {d_ins.mean():+.3f} (SD {d_ins.std():.3f}); IG better in {(d_ins>0).sum()} of {len(df)} seeds")

 seed  test_acc  delete_IG  delete_pressure  delete_random  insert_IG  insert_pressure  insert_random
    1    0.6964     0.2414           0.2539         0.6455     0.3018           0.2954         0.0675
    2    0.6925     0.2644           0.2754         0.6432     0.3138           0.3254         0.0710
    3    0.7321     0.2659           0.2769         0.6808     0.3661           0.3477         0.0778
    4    0.7341     0.2462           0.2486         0.6780     0.3219           0.3306         0.0766
    5    0.7143     0.2492           0.2667         0.6724     0.3643           0.3538         0.0928

MEAN +/- SD over seeds
      test_acc  delete_IG  delete_pressure  delete_random  insert_IG  insert_pressure  insert_random
mean     0.714      0.253            0.264          0.664      0.334            0.331          0.077
std      0.019      0.011            0.013          0.018      0.030            0.023          0.010

Deletion: pressure AUC minus IG AUC = +0.011 (SD 0.005); IG 

## 2. Stability of explanations across independently trained models
Uses `src/stability.py`.

In [ ]:
!python -m src.stability

model seed 1: accuracy on the 200 explained trials = 0.740
model seed 2: accuracy on the 200 explained trials = 0.730
model seed 3: accuracy on the 200 explained trials = 0.715
model seed 4: accuracy on the 200 explained trials = 0.760
model seed 5: accuracy on the 200 explained trials = 0.730

STABILITY over 5 models, 200 trials (10 model pairs)
  IG sensor ranking, same trial, different models (Spearman)  : 0.99
  IG top-10% sensors overlap (Jaccard)                        : 0.66
  control: different trial of the same activity (Spearman)   : 0.24
  IG vs raw mean pressure, within one model (Spearman)        : 0.99
  CAM time curves, same trial, different models (Pearson)     : 0.69


In [ ]:
!cat results/xai_stability.csv

models,trials,IG_spearman_same_trial,IG_top10_jaccard,IG_spearman_other_trial_same_class,IG_vs_pressure_spearman,CAM_pearson_same_trial
5,200,0.993,0.658,0.241,0.995,0.695


## Self-check and download

In [ ]:
import pandas as pd
from pathlib import Path
R = Path("results")
ok = {"5-split CSV exists with 5 seeds": (R/"xai_faithfulness_multiseed.csv").exists() and len(pd.read_csv(R/"xai_faithfulness_multiseed.csv")) == 5,
      "stability CSV exists": (R/"xai_stability.csv").exists()}
if ok["stability CSV exists"]:
    s = pd.read_csv(R/"xai_stability.csv").iloc[0]; print("stability:", s.round(3).to_dict())
for k, v in ok.items(): print("PASS" if v else "FAIL", "-", k)
assert all(ok.values()), "A check failed - read the FAIL lines above"
print("\nNotebook 04 self-check: ALL PASS")

stability: {'models': 5.0, 'trials': 200.0, 'IG_spearman_same_trial': 0.993, 'IG_top10_jaccard': 0.658, 'IG_spearman_other_trial_same_class': 0.241, 'IG_vs_pressure_spearman': 0.995, 'CAM_pearson_same_trial': 0.695}
PASS - 5-split CSV exists with 5 seeds
PASS - stability CSV exists

Notebook 04 self-check: ALL PASS


In [ ]:
!zip -j results_04.zip results/xai_faithfulness_multiseed.csv results/xai_stability.csv
from google.colab import files
files.download("results_04.zip")

  adding: xai_faithfulness_multiseed.csv (deflated 48%)
  adding: xai_stability.csv (deflated 31%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>